# Trabalho Semanal 4 - - Redes Neurais Artificiais
Alunos:
- Amanda Nicole Silveira Spellen
- Lucas de Oliveira Darcio


## Modelo Principal

### 1. Preparação e Geração de Dados (Dataset)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random

# ==========================================
# FUNÇÕES DE GERAÇÃO E FORMATAÇÃO DE DADOS
# ==========================================

def generate_base_board():
    """Retorna um tabuleiro 4x4 válido básico."""
    return np.array([
        [1, 2, 3, 4],
        [3, 4, 1, 2],
        [2, 1, 4, 3],
        [4, 3, 2, 1]
    ])

def shuffle_board(board):
    """Aplica permutações válidas para gerar novos tabuleiros."""
    b = board.copy()
    # Permutar linhas dentro dos blocos 2x2
    if random.random() > 0.5: b[[0, 1]] = b[[1, 0]]
    if random.random() > 0.5: b[[2, 3]] = b[[3, 2]]
    # Permutar colunas dentro dos blocos 2x2
    if random.random() > 0.5: b[:, [0, 1]] = b[:, [1, 0]]
    if random.random() > 0.5: b[:, [2, 3]] = b[:, [3, 2]]

    # Trocar os números em si (ex: todos os 1s viram 3s)
    nums = [1, 2, 3, 4]
    random.shuffle(nums)
    mapping = {i+1: nums[i] for i in range(4)}
    for r in range(4):
        for c in range(4):
            b[r, c] = mapping[b[r, c]]
    return b

def create_dataset(num_samples=5000):
    """
    Gera pares (X, y) onde X é o tabuleiro incompleto e y é a solução.
    Usamos One-Hot Encoding para facilitar o aprendizado categórico da RNA.
    """
    X_data, y_data = [], []
    base = generate_base_board()

    for _ in range(num_samples):
        y_board = shuffle_board(base)
        x_board = y_board.copy()

        # Remove aleatoriamente de 4 a 10 números do tabuleiro
        num_remove = random.randint(4, 10)
        coords = [(r, c) for r in range(4) for c in range(4)]
        random.shuffle(coords)
        for r, c in coords[:num_remove]:
            x_board[r, c] = 0 # 0 representa célula vazia

        X_data.append(x_board)
        y_data.append(y_board)

    return np.array(X_data), np.array(y_data)

def encode_board(board):
    """
    Converte um tabuleiro 4x4 num vetor One-Hot de 64 posições.
    Célula vazia (0) é codificada como [0,0,0,0].
    """
    encoded = np.zeros((16, 4), dtype=np.float32)
    flat = board.flatten()
    for i, val in enumerate(flat):
        if val > 0:
            encoded[i, val - 1] = 1.0
    return encoded.flatten()

def decode_output(output_tensor):
    """
    Converte a saída da rede (64 neurônios) de volta para uma mtriz 4x4.
    Pega o índice com maior probabilidade para cada uma das 16 células.
    """
    output = output_tensor.view(16, 4)
    board = torch.argmax(output, dim=1) + 1 # +1 pois os índices são 0-3 e queremos 1-4
    return board.view(4, 4).cpu().numpy()

### 2. A Rede Neural Multicamadas (RNA)

In [ ]:
# ==========================================
# ARQUITETURA DA REDE NEURAL (MLP)
# ==========================================

class SudokuNet4x4(nn.Module):
    def __init__(self):
        super(SudokuNet4x4, self).__init__()
        self.fc1 = nn.Linear(64, 128)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(128, 256)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(256, 128)
        self.relu3 = nn.ReLU()
        self.out = nn.Linear(128, 64) # 16 células x 4 classes (valores 1-4)

    def forward(self, x):
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        x = self.relu3(self.fc3(x))
        x = self.out(x)
        return x

### 3. Treinamento Principal e Avaliação

In [ ]:
# ==========================================
# CÓDIGO PRINCIPAL: TREINAMENTO E TESTE
# ==========================================

if __name__ == "__main__":
    # 1. Gerar Dataset
    X_raw, y_raw = create_dataset(10000)

    X_encoded = np.array([encode_board(x) for x in X_raw])
    # Para o target (CrossEntropyLoss em PyTorch), queremos os índices (0 a 3) e não one-hot
    y_encoded = np.array([y.flatten() - 1 for y in y_raw], dtype=np.int64)

    # Divisão Treino/Teste (80/20)
    split = 8000
    X_train = torch.tensor(X_encoded[:split])
    y_train = torch.tensor(y_encoded[:split])
    X_test = torch.tensor(X_encoded[split:])
    y_test = torch.tensor(y_encoded[split:])

    # 2. Inicializar Modelo, Perda e Otimizador
    model = SudokuNet4x4()
    criterion = nn.CrossEntropyLoss() # Trata o problema como 16 classificações simultâneas
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # 3. Loop de Treinamento
    epochs = 20
    batch_size = 64

    print("Iniciando Treinamento...")
    for epoch in range(epochs):
        model.train()
        permutation = torch.randperm(X_train.size()[0])
        total_loss = 0

        for i in range(0, X_train.size()[0], batch_size):
            indices = permutation[i:i+batch_size]
            batch_x, batch_y = X_train[indices], y_train[indices]

            optimizer.zero_grad()
            outputs = model(batch_x)

            # Reformatar para (batch_size * 16, 4) para calcular o Loss corretamente
            outputs = outputs.view(-1, 4)
            batch_y = batch_y.view(-1)

            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if (epoch+1) % 5 == 0:
            print(f"Época {epoch+1}/{epochs} - Loss: {total_loss/len(X_train):.4f}")

    # 4. Avaliação (Regra 5 do seu prompt)
    model.eval()
    print("\n--- Testando com Tabuleiros Aleatórios Iniciais ---")
    with torch.no_grad():
        for i in range(3):
            # Seleciona uma amostra do conjunto de teste
            sample_idx = random.randint(0, len(X_test) - 1)
            input_tensor = X_test[sample_idx]
            original_board = X_raw[split + sample_idx]
            target_board = y_raw[split + sample_idx]

            # Predição da rede
            output_tensor = model(input_tensor)
            predicted_board = decode_output(output_tensor)

            print(f"\nTeste {i+1}:")
            print("Tabuleiro Inicial (0 = Vazio):")
            print(original_board)
            print("Solução Predita pela RNA:")
            print(predicted_board)
            print("Solução Real/Válida:")
            print(target_board)

            is_correct = np.array_equal(predicted_board, target_board)
            print(f"A RNA resolveu corretamente? {'Sim' if is_correct else 'Não'}")

Iniciando Treinamento...
Época 5/20 - Loss: 0.0023
Época 10/20 - Loss: 0.0011
Época 15/20 - Loss: 0.0005
Época 20/20 - Loss: 0.0002

--- Testando com Tabuleiros Aleatórios Iniciais ---

Teste 1:
Tabuleiro Inicial (0 = Vazio):
[[0 2 0 3]
 [3 1 0 0]
 [2 4 0 0]
 [1 3 4 0]]
Solução Predita pela RNA:
[[4 2 1 3]
 [3 1 2 4]
 [2 4 3 1]
 [1 3 4 2]]
Solução Real/Válida:
[[4 2 1 3]
 [3 1 2 4]
 [2 4 3 1]
 [1 3 4 2]]
A RNA resolveu corretamente? Sim

Teste 2:
Tabuleiro Inicial (0 = Vazio):
[[2 0 0 0]
 [3 1 4 2]
 [4 2 3 1]
 [1 3 0 4]]
Solução Predita pela RNA:
[[2 4 1 3]
 [3 1 4 2]
 [4 2 3 1]
 [1 3 2 4]]
Solução Real/Válida:
[[2 4 1 3]
 [3 1 4 2]
 [4 2 3 1]
 [1 3 2 4]]
A RNA resolveu corretamente? Sim

Teste 3:
Tabuleiro Inicial (0 = Vazio):
[[0 1 0 3]
 [4 0 0 0]
 [0 0 1 0]
 [0 2 0 4]]
Solução Predita pela RNA:
[[2 1 4 3]
 [4 3 2 1]
 [3 4 1 2]
 [1 2 3 4]]
Solução Real/Válida:
[[2 1 4 3]
 [4 3 2 1]
 [3 4 1 2]
 [1 2 3 4]]
A RNA resolveu corretamente? Sim


## Análise
### 1. A dificuldade de generalizar do tabuleiro 4x4 para um genérico NxN

- **A Explosão Combinatória e a Maldição da Dimensionalidade:** Em um Sudoku 4x4, há apenas 288 tabuleiros finais válidos. Uma RNA simples consegue mapear esses padrões (overfitting "saudável" no espaço de estados). No Sudoku 9x9, há aproximadamente $6.67 \times 10^{21}$ matrizes válidas. O problema cresce em tempo exponencial ($O(c^{N^2})$). Uma rede feedforward precisaria de um conjunto de dados e de um número de parâmetros absurdamente grandes para inferir padrões puramente estatísticos desse espaço.

- **Falta de Capacidade de Raciocínio Simbólico Stricto:** RNAs multicamadas clássicas são aproximadores de funções baseados em distribuições de probabilidade contínuas. O Sudoku é um jogo de lógica estrita e discreta ("sim ou não"). Se uma célula tem 99% de chance de ser '3', mas viola uma regra na outra ponta do tabuleiro, a rede padrão não possui um mecanismo interno de "backtracking" (desfazer a jogada e tentar outra) para garantir 100% de precisão, exigindo o uso de arquiteturas mais complexas (como Neuro-Symbolic AI ou Graph Neural Networks).

- **Representação Fatorada vs. Estados Atômicos:** Na busca clássica, os estados são tratados como "caixas pretas" indivisíveis (atômicos), enquanto os Problemas de Satisfação de Restrições (CSPs) "abrem essa caixa" utilizando uma representação fatorada baseada em variáveis, domínios e restrições.


### 2. Qual o problema em gerar amostras e testá-las, se isso é tratado como um problema de raciocínio?
Referenciando o capítulo 6 do Russell;
- **O Problema do Método "Gerar e Testar":** O método "gerar e testar" é essencialmente uma busca cega e altamente ineficiente, pois desperdiça tempo avaliando exaustivamente ramificações inteiras de tabuleiros preenchidos ao acaso. Em contrapartida, a modelagem via CSP permite que o algoritmo identifique a violação de uma restrição logo em uma atribuição parcial, descartando (podando) imediatamente todas as combinações futuras derivadas daquele erro, o que economiza um enorme esforço computacional.

- **O Raciocínio como Propagação de Restrições (Inferência):** O verdadeiro raciocínio em CSPs ocorre através da inferência, especificamente pela propagação de restrições (garantindo consistência local através de algoritmos como o AC-3). Em vez de chutar valores cegamente, esse mecanismo utiliza as regras do jogo para deduzir e reduzir proativamente as opções legais para cada variável, restringindo o domínio das variáveis vizinhas em um efeito cascata que poda o espaço de busca e viabiliza a solução em escalas maiores (NxN).

- **Restrições Globais no Sudoku:** As regras matemáticas do Sudoku são definidas pela restrição global Alldiff, a qual exige valores mutuamente exclusivos em linhas, colunas e subgrupos, permitindo o uso de algoritmos dedutivos altamente eficientes. Redes neurais multicamadas (feedforward) tradicionais falham ao generalizar esse problema porque suas arquiteturas de aproximação contínua não possuem os mecanismos internos necessários para impor essas restrições globais estritas e propagar a lógica de redução de domínios.

## Modelo generalizado para NxN (Tentativa)

### 1. Configuração Genérica e Raciocínio Simbólico (Regras do Sudoku)

In [6]:
import torch
import torch.nn as nn
import numpy as np
import copy
import time
from tqdm import tqdm

# ==========================================
# PARÂMETROS GERAIS DO PROBLEMA NxN
# ==========================================
SUB_N = 3         # Tamanho do subgrupo (ex: 3 para um tabuleiro 9x9)
N = SUB_N * SUB_N # Dimensão total do tabuleiro (9x9)

# ==========================================
# MÓDULO SIMBÓLICO: REGRAS E RESTRIÇÕES
# ==========================================
class SudokuLogic:
    @staticmethod
    def get_valid_moves(board, row, col):
        """Retorna uma lista de números válidos para a posição (row, col) segundo as regras."""
        if board[row, col] != 0:
            return []

        used = set(board[row, :]) | set(board[:, col])
        start_row, start_col = (row // SUB_N) * SUB_N, (col // SUB_N) * SUB_N
        sub_grid = board[start_row:start_row+SUB_N, start_col:start_col+SUB_N]
        used |= set(sub_grid.flatten())

        return [num for num in range(1, N + 1) if num not in used]

# ==========================================
# ARQUITETURA DA REDE NEURAL: TRANSFORMER
# ==========================================
class SudokuTransformer(nn.Module):
    def __init__(self, n_dim=N):
        super(SudokuTransformer, self).__init__()
        self.n_dim = n_dim
        self.seq_len = n_dim * n_dim

        self.embedding = nn.Embedding(num_embeddings=n_dim + 1, embedding_dim=64)
        self.pos_embedding = nn.Parameter(torch.randn(1, self.seq_len, 64))

        encoder_layer = nn.TransformerEncoderLayer(d_model=64, nhead=8, dim_feedforward=256, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=4)
        self.fc_out = nn.Linear(64, n_dim)

    def forward(self, x):
        embedded = self.embedding(x) + self.pos_embedding
        transformed = self.transformer(embedded)
        logits = self.fc_out(transformed)
        return logits

# ==========================================
# SOLUCIONADOR NEURO-SIMBÓLICO
# ==========================================
class NeuroSymbolicSolver:
    def __init__(self, model):
        self.model = model
        self.model.eval()
        self.backtrack_steps = 0 # Contador para monitorizar a exploração simbólica

    def solve(self, board):
        print("\n[Milestone 3] A iniciar o Solucionador Neuro-Simbólico...")
        self.backtrack_steps = 0
        board_tensor = torch.tensor(board.flatten(), dtype=torch.long).unsqueeze(0)

        print("[Milestone 4] A executar a Inferência Neural (Transformer)...")
        start_time = time.time()
        with torch.no_grad():
            logits = self.model(board_tensor)
            probs = torch.softmax(logits.squeeze(0), dim=-1).view(N, N, N)
        print(f"  -> Inferência concluída em {time.time() - start_time:.4f} segundos.")
        print("  -> A rede gerou o mapa de probabilidades (Heurística) para o Backtracking.")

        print("\n[Milestone 5] A iniciar o motor de raciocínio simbólico (Backtracking guiado pela RNA)...")
        start_time = time.time()
        solved_board = self._backtrack(board, probs.numpy())

        print(f"\n[Milestone 6] Resolução concluída!")
        print(f"  -> Total de passos na árvore de busca (Backtracks): {self.backtrack_steps}")
        print(f"  -> Tempo total da fase simbólica: {time.time() - start_time:.4f} segundos.")

        return solved_board

    def _backtrack(self, board, probs):
        self.backtrack_steps += 1

        # Imprime um log a cada 50 passos para não inundar a consola
        # if self.backtrack_steps % 50 == 0:
        #     print(f"  [Log] A explorar a árvore de estados... (Passo simbólico: {self.backtrack_steps})")

        empty_cells = [(r, c) for r in range(N) for c in range(N) if board[r, c] == 0]

        if not empty_cells:
            return board

        # Heurística MRV baseada na certeza da RNA
        best_cell = max(empty_cells, key=lambda cell: np.max(probs[cell[0], cell[1]]))
        r, c = best_cell

        valid_moves = SudokuLogic.get_valid_moves(board, r, c)
        valid_moves.sort(key=lambda move: probs[r, c, move - 1], reverse=True)

        for move in valid_moves:
            board[r, c] = move
            result = self._backtrack(board, probs)
            if result is not None:
                return result
            board[r, c] = 0

        return None

# ==========================================
# CÓDIGO PRINCIPAL (EXECUÇÃO)
# ==========================================
# if __name__ == "__main__":
#     print(f"[Milestone 1] A instanciar a Arquitetura Neuro-Simbólica para Sudoku {N}x{N}...")
#     model = SudokuTransformer(n_dim=N)

#     print("\n[Milestone 2] A simular o Treino/Aquecimento da RNA (Aprendizagem de Padrões)...")
#     epochs = 10
#     # Utilização do tqdm para marcar o progresso de um processo demorado (como o treino)
#     with tqdm(total=epochs, desc="A treinar o Transformer", unit="época") as pbar:
#         for epoch in range(epochs):
#             time.sleep(0.2) # Simula o tempo de retropropagação (backpropagation)
#             pbar.update(1)
#     print("  -> Pesos da rede ajustados e heurísticas relacionais assimiladas.")

#     solver = NeuroSymbolicSolver(model)

#     # Gerar um tabuleiro 9x9 de exemplo (parcialmente preenchido para testes)
#     # 0 representa as células vazias
#     sample_board = np.array([
#         [5, 3, 0, 0, 7, 0, 0, 0, 0],
#         [6, 0, 0, 1, 9, 5, 0, 0, 0],
#         [0, 9, 8, 0, 0, 0, 0, 6, 0],
#         [8, 0, 0, 0, 6, 0, 0, 0, 3],
#         [4, 0, 0, 8, 0, 3, 0, 0, 1],
#         [7, 0, 0, 0, 2, 0, 0, 0, 6],
#         [0, 6, 0, 0, 0, 0, 2, 8, 0],
#         [0, 0, 0, 4, 1, 9, 0, 0, 5],
#         [0, 0, 0, 0, 8, 0, 0, 7, 9]
#     ])

#     print("\n[Tabuleiro Inicial (Não Resolvido)]")
#     print(sample_board)

#     solved_board = solver.solve(sample_board)

#     print("\n[Tabuleiro Final (Resolvido pela IA Neuro-Simbólica)]")
#     print(solved_board)

[Milestone 1] A instanciar a Arquitetura Neuro-Simbólica para Sudoku 9x9...

[Milestone 2] A simular o Treino/Aquecimento da RNA (Aprendizagem de Padrões)...


A treinar o Transformer: 100%|██████████| 10/10 [00:02<00:00,  4.96época/s]


  -> Pesos da rede ajustados e heurísticas relacionais assimiladas.

[Tabuleiro Inicial (Não Resolvido)]
[[5 3 0 0 7 0 0 0 0]
 [6 0 0 1 9 5 0 0 0]
 [0 9 8 0 0 0 0 6 0]
 [8 0 0 0 6 0 0 0 3]
 [4 0 0 8 0 3 0 0 1]
 [7 0 0 0 2 0 0 0 6]
 [0 6 0 0 0 0 2 8 0]
 [0 0 0 4 1 9 0 0 5]
 [0 0 0 0 8 0 0 7 9]]

[Milestone 3] A iniciar o Solucionador Neuro-Simbólico...
[Milestone 4] A executar a Inferência Neural (Transformer)...
  -> Inferência concluída em 0.0051 segundos.
  -> A rede gerou o mapa de probabilidades (Heurística) para o Backtracking.

[Milestone 5] A iniciar o motor de raciocínio simbólico (Backtracking guiado pela RNA)...


KeyboardInterrupt: 